# 07. Ridge com dataset enriquecido pelas tabelas auxiliares

**Objetivo:** repetir o teste do notebook 06 (Ridge com colunas originais), agora com um dataset **enriquecido**: partindo dos contratos selecionados e juntando, pela chave correspondente, as tabelas de parcelas, liberações e aditivos.

**Granularidade:** a unidade de análise continua sendo **1 linha por contrato** (48 linhas), porque o alvo é por contrato. As tabelas auxiliares têm várias linhas por contrato, então cada uma é resumida por contrato antes do join (contagem, soma ou contagem por categoria das colunas originais). Não há razões, logs, datas derivadas nem outras features construídas.

- **Alvo:** `max(0, duração real − prazo ajustado)`, igual aos notebooks 05 e 06 (duração real = `data_recebimento_provisorio − data_ordem_servico`; prazo ajustado inclui os aditivos de prazo).
- **Baseline:** mediana do atraso no treino. **Modelo:** Ridge.
- **Validação:** 4 janelas temporais expanding (20→5, 25→5, 30→5, 35→5) nos 40 contratos mais antigos e teste final nos 8 mais recentes, por `data_recebimento_provisorio`.

> **Atenção:** as tabelas auxiliares descrevem a obra **inteira** (parcelas, liberações e aditivos que só existem durante ou depois da execução). Este notebook mede o quanto esses dados *explicam* o atraso; ele não representa uma previsão feita na ordem de serviço. Por isso o notebook mostra, na seção 8, o efeito de cada grupo de tabelas.

In [1]:
from pathlib import Path
import unicodedata
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

N_TEST = 8
MIN_TRAIN = 20
BLOCK = 5
N_FOLDS = 4
RIDGE_ALPHA = 1.0

# Se os CSVs estiverem em outra pasta, defina DATA_DIR manualmente.
DATA_DIR = None

## 1. Carga dos dados e cobertura das tabelas auxiliares

In [2]:
FILES = {
    "contratos": "vw_modelagem_contratos_filtros.csv",
    "aditivos_prazo": "vw_taditivoprazo.csv",
    "aditivos_valor": "vw_taditivovalor.csv",
    "liberacoes": "vw_tparcela_liberacao.csv",
    "parcelas": "vw_bi_parcelas.csv",
}


def find_data_dir():
    if DATA_DIR is not None:
        return Path(DATA_DIR)
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "data", cwd / "data" / "raw", Path("/mnt/user-data/uploads")]
    for parent in cwd.parents:
        candidates += [parent / "data", parent / "data" / "raw"]
    for folder in candidates:
        if (folder / FILES["contratos"]).exists():
            return folder
    raise FileNotFoundError("CSVs não encontrados; defina DATA_DIR.")


base = find_data_dir()


def read(name, **kw):
    return pd.read_csv(base / FILES[name], encoding="utf-8-sig", **kw)


contratos = read("contratos", parse_dates=["data_ordem_servico", "data_recebimento_provisorio"])
aditivos_prazo = read("aditivos_prazo", parse_dates=["Data_termino_anterior", "Data_termino_nova"])
aditivos_valor = read("aditivos_valor")
liberacoes = read("liberacoes")
parcelas = read("parcelas")

assert contratos["idContrato"].is_unique
ids = set(contratos["idContrato"])

# Chave de cada tabela para chegar ao contrato.
CHAVES = {
    "parcelas": (parcelas, "id_contrato"),
    "liberacoes": (liberacoes, "id_Contrato"),
    "aditivos_prazo": (aditivos_prazo, "idContrato"),
    "aditivos_valor": (aditivos_valor, "idContrato"),
}

cobertura = pd.DataFrame([
    {
        "tabela": nome,
        "chave": chave,
        "linhas_total": len(t),
        "linhas_na_selecao": int(t[chave].isin(ids).sum()),
        "contratos_cobertos": t.loc[t[chave].isin(ids), chave].nunique(),
        "contratos_sem_linha": len(ids) - t.loc[t[chave].isin(ids), chave].nunique(),
    }
    for nome, (t, chave) in CHAVES.items()
])

print("Pasta de dados:", base)
print("Contratos selecionados:", len(ids))
display(cobertura)

# Integridade entre parcelas e liberações (chave id_Parcela = id_parcela)
lib_sel = liberacoes[liberacoes["id_Contrato"].isin(ids)]
par_sel = parcelas[parcelas["id_contrato"].isin(ids)]
print("Liberações cujo id_Parcela existe em parcelas:", round(lib_sel["id_Parcela"].isin(par_sel["id_parcela"]).mean(), 3))

Pasta de dados: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw
Contratos selecionados: 48


,tabela,chave,linhas_total,linhas_na_selecao,contratos_cobertos,contratos_sem_linha
0,parcelas,id_contrato,13010,897,47,1
1,liberacoes,id_Contrato,11918,886,47,1
2,aditivos_prazo,idContrato,245,85,37,11
3,aditivos_valor,idContrato,271,134,38,10


Liberações cujo id_Parcela existe em parcelas: 1.0


## 2. Alvo com ajuste por aditivos de prazo

Fim original = `data_ordem_servico + prazo_total`. Com aditivo, o fim ajustado é a maior `Data_termino_nova` do contrato (término vigente).

In [3]:
df = contratos.copy()

df["fim_original"] = df["data_ordem_servico"] + pd.to_timedelta(df["prazo_total"], unit="D")

fim_aditado = aditivos_prazo.groupby("idContrato")["Data_termino_nova"].max().rename("fim_aditado")
df = df.merge(fim_aditado, on="idContrato", how="left")

df["fim_ajustado"] = df[["fim_original", "fim_aditado"]].max(axis=1)
df["prazo_ajustado"] = (df["fim_ajustado"] - df["data_ordem_servico"]).dt.days
df["duracao_real"] = (df["data_recebimento_provisorio"] - df["data_ordem_servico"]).dt.days

df["atraso_bruto"] = df["duracao_real"] - df["prazo_ajustado"]
df["target_dias_atraso"] = df["atraso_bruto"].clip(lower=0)

print("Contratos com atraso:", int((df["target_dias_atraso"] > 0).sum()), "de", len(df))

Contratos com atraso: 17 de 48


## 3. Construção do dataset enriquecido

Cada tabela auxiliar é filtrada pelos contratos selecionados, resumida **por contrato** e ligada ao dataset com `left join` pela chave. Contratos sem linhas na tabela auxiliar recebem 0 (não têm parcelas, liberações ou aditivos).

| Tabela | Chave | Colunas originais usadas | Resumo por contrato |
|---|---|---|---|
| `vw_bi_parcelas` | `id_contrato` | `id_parcela`, `tipo_de_parcela`, `valor_reajustado`, `valor_ja_liberado` | nº de parcelas distintas por tipo; soma dos valores (uma linha por `id_parcela`). `valor_ja_liberado` é 0 em todos os contratos e é removida |
| `vw_tparcela_liberacao` | `id_Contrato` | `id_Parcela`, `Valor_total_liberado`, `Abono_atraso`, `Atraso_considerado` | nº de liberações; soma dos três campos |
| `vw_taditivoprazo` | `idContrato` | `Aditado_em_dias` | nº de aditivos; soma dos dias |
| `vw_taditivovalor` | `idContrato` | (sem valor numérico na tabela) | nº de aditivos |

Colunas de data, status e texto das tabelas auxiliares ficam de fora: as datas só virariam feature depois de derivações, e os status são quase constantes.

In [4]:
def slug(texto):
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return texto.lower().replace(" ", "_")


# --- parcelas (uma linha por id_parcela: há linhas repetidas por execução parcial)
p = parcelas[parcelas["id_contrato"].isin(ids)].drop_duplicates("id_parcela")

parc_tipos = p.pivot_table(
    index="id_contrato", columns="tipo_de_parcela", values="id_parcela", aggfunc="nunique", fill_value=0
)
parc_tipos.columns = ["parc_qtd_" + slug(c) for c in parc_tipos.columns]

parc_valores = p.groupby("id_contrato").agg(
    parc_valor_reajustado_soma=("valor_reajustado", "sum"),
    parc_valor_ja_liberado_soma=("valor_ja_liberado", "sum"),
)
parc_agg = parc_tipos.join(parc_valores)

# --- liberações
lib_agg = (
    liberacoes[liberacoes["id_Contrato"].isin(ids)]
    .groupby("id_Contrato")
    .agg(
        lib_qtd=("id_Parcela", "size"),
        lib_valor_total_liberado_soma=("Valor_total_liberado", "sum"),
        lib_abono_atraso_soma=("Abono_atraso", "sum"),
        lib_atraso_considerado_soma=("Atraso_considerado", "sum"),
    )
)

# --- aditivos de prazo e de valor
adp_agg = (
    aditivos_prazo[aditivos_prazo["idContrato"].isin(ids)]
    .groupby("idContrato")
    .agg(adp_qtd=("Aditado_em_dias", "size"), adp_dias_aditados_soma=("Aditado_em_dias", "sum"))
)
adv_agg = (
    aditivos_valor[aditivos_valor["idContrato"].isin(ids)]
    .groupby("idContrato")
    .agg(adv_qtd=("idContrato", "size"))
)

n_antes = len(df)

enriquecido = df
for agg in [parc_agg, lib_agg, adp_agg, adv_agg]:
    enriquecido = enriquecido.merge(agg, left_on="idContrato", right_index=True, how="left")

COLUNAS_AUX = [c for agg in [parc_agg, lib_agg, adp_agg, adv_agg] for c in agg.columns]
enriquecido[COLUNAS_AUX] = enriquecido[COLUNAS_AUX].fillna(0)

# Colunas sem variação nos 48 contratos não servem como feature.
constantes = [c for c in COLUNAS_AUX if enriquecido[c].nunique() == 1]
print("Colunas auxiliares constantes (removidas):", constantes)
enriquecido = enriquecido.drop(columns=constantes)
COLUNAS_AUX = [c for c in COLUNAS_AUX if c not in constantes]
parc_agg, lib_agg, adp_agg, adv_agg = [a.drop(columns=[c for c in constantes if c in a.columns]) for a in (parc_agg, lib_agg, adp_agg, adv_agg)]

# O join não pode duplicar nem perder contratos.
assert len(enriquecido) == n_antes == 48
assert enriquecido["idContrato"].is_unique

print("Colunas auxiliares adicionadas:", len(COLUNAS_AUX))
display(enriquecido[["idContrato"] + COLUNAS_AUX].head())
display(enriquecido[COLUNAS_AUX].describe().T[["mean", "min", "max"]])

Colunas auxiliares constantes (removidas): ['parc_valor_ja_liberado_soma']
Colunas auxiliares adicionadas: 14


,idContrato,parc_qtd_aditivo_historico,parc_qtd_aditivo_mercado,parc_qtd_reajuste,parc_qtd_servicos_originais,parc_qtd_supressao_historico,parc_qtd_supressao_mercado,parc_valor_reajustado_soma,lib_qtd,lib_valor_total_liberado_soma,lib_abono_atraso_soma,lib_atraso_considerado_soma,adp_qtd,adp_dias_aditados_soma,adv_qtd
0,11,1.000,1.000,2.000,5.000,1.000,1.000,"497,109.910",9.000,"390,675.380",0.000,50.000,0.000,0.000,2.000
1,12,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2,18,1.000,1.000,0.000,4.000,1.000,0.000,"290,075.080",8.000,"290,075.080",0.000,0.000,1.000,30.000,1.000
3,29,6.000,3.000,4.000,16.000,5.000,0.000,"711,018.461",38.000,"713,630.840","1,606.000",40.000,4.000,178.000,5.000
4,30,0.000,0.000,0.000,3.000,0.000,0.000,"393,214.860",4.000,"393,214.860",0.000,0.000,0.000,0.000,0.000


,mean,min,max
parc_qtd_aditivo_historico,2.208,0.000,13.000
parc_qtd_aditivo_mercado,1.917,0.000,11.000
parc_qtd_reajuste,1.125,0.000,8.000
parc_qtd_servicos_originais,8.625,0.000,32.000
parc_qtd_supressao_historico,1.667,0.000,12.000
parc_qtd_supressao_mercado,0.042,0.000,1.000
parc_valor_reajustado_soma,"3,402,907.196",0.000,"30,141,450.588"
lib_qtd,18.458,0.000,110.000
lib_valor_total_liberado_soma,"2,927,455.364",0.000,"22,804,713.100"
lib_abono_atraso_soma,193.604,0.000,"1,606.000"


## 4. Seleção de colunas

- **Contrato:** mesmas colunas do notebook 06 (`Perc_desconto`, `Classificacao_contrato`, `id_setor_divisao`). Continuam fora `prazo_total`, `valor_total_contrato`, `Valor_licitado`, `Flag_ativo_emissao_OS`, identificadores, constantes, datas, texto e informações pós-obra.
- **Tabelas auxiliares:** todas as colunas resumidas na seção 3, organizadas em grupos.

O grupo diz **quando a informação passa a existir**, o que importa para interpretar o resultado.

In [5]:
CONTRATO_NUM = ["Perc_desconto"]
CONTRATO_CAT = ["Classificacao_contrato", "id_setor_divisao"]

GRUPOS_AUX = {
    "parcelas": list(parc_agg.columns),
    "aditivos_valor": list(adv_agg.columns),
    "aditivos_prazo": list(adp_agg.columns),
    "liberacoes": list(lib_agg.columns),
}

QUANDO_EXISTE = {
    "contrato": "na ordem de serviço",
    "parcelas": "parcelas originais: no início; aditivos, supressões e reajustes: durante a obra",
    "aditivos_valor": "durante a obra",
    "aditivos_prazo": "durante a obra (também definem o prazo ajustado do alvo)",
    "liberacoes": "durante e depois da obra (inclui atraso considerado por parcela)",
}

NUMERIC_FEATURES = CONTRATO_NUM + [c for cols in GRUPOS_AUX.values() for c in cols]
CATEGORICAL_FEATURES = CONTRATO_CAT
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

resumo = pd.DataFrame(
    [{"grupo": "contrato", "colunas": ", ".join(CONTRATO_NUM + CONTRATO_CAT), "quando_existe": QUANDO_EXISTE["contrato"]}]
    + [{"grupo": g, "colunas": ", ".join(cols), "quando_existe": QUANDO_EXISTE[g]} for g, cols in GRUPOS_AUX.items()]
)
print(f"{len(FEATURES)} features ({len(NUMERIC_FEATURES)} numéricas, {len(CATEGORICAL_FEATURES)} categóricas)")
display(resumo)

17 features (15 numéricas, 2 categóricas)


,grupo,colunas,quando_existe
0,contrato,"Perc_desconto, Classificacao_contrato, id_setor_divisao",na ordem de serviço
1,parcelas,"parc_qtd_aditivo_historico, parc_qtd_aditivo_mercado, parc_qtd_reajuste, parc_qtd_servicos_originais, parc...","parcelas originais: no início; aditivos, supressões e reajustes: durante a obra"
2,aditivos_valor,adv_qtd,durante a obra
3,aditivos_prazo,"adp_qtd, adp_dias_aditados_soma",durante a obra (também definem o prazo ajustado do alvo)
4,liberacoes,"lib_qtd, lib_valor_total_liberado_soma, lib_abono_atraso_soma, lib_atraso_considerado_soma",durante e depois da obra (inclui atraso considerado por parcela)


In [12]:
print(FEATURES)

['Perc_desconto', 'parc_qtd_aditivo_historico', 'parc_qtd_aditivo_mercado', 'parc_qtd_reajuste', 'parc_qtd_servicos_originais', 'parc_qtd_supressao_historico', 'parc_qtd_supressao_mercado', 'parc_valor_reajustado_soma', 'adv_qtd', 'adp_qtd', 'adp_dias_aditados_soma', 'lib_qtd', 'lib_valor_total_liberado_soma', 'lib_abono_atraso_soma', 'lib_atraso_considerado_soma', 'Classificacao_contrato', 'id_setor_divisao']


## 5. Separação desenvolvimento × teste final

In [6]:
enriquecido = enriquecido.sort_values(["data_recebimento_provisorio", "idContrato"]).reset_index(drop=True)

dev = enriquecido.iloc[:-N_TEST].copy()
test = enriquecido.iloc[-N_TEST:].copy()

assert dev["data_recebimento_provisorio"].max() <= test["data_recebimento_provisorio"].min()

folds = []
for k in range(N_FOLDS):
    n_train = MIN_TRAIN + k * BLOCK
    folds.append({"fold": k + 1, "train": dev.iloc[:n_train], "val": dev.iloc[n_train:n_train + BLOCK]})

print("Contratos de teste:", test["idContrato"].tolist())

Contratos de teste: [173, 201, 287, 283, 342, 317, 254, 377]


## 6. Modelos, métricas e validação temporal

- **Baseline:** mediana do treino (`DummyRegressor`).
- **Ridge:** numéricas padronizadas e categóricas em one-hot; previsões negativas viram zero.

In [7]:
def make_baseline():
    return DummyRegressor(strategy="median")


def make_ridge(num, cat):
    transformers = []
    if num:
        transformers.append(("num", StandardScaler(), num))
    if cat:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat))
    return Pipeline([("prep", ColumnTransformer(transformers)), ("model", Ridge(alpha=RIDGE_ALPHA))])


def predict(model, frame, cols):
    return np.maximum(0, model.predict(frame[cols]))


def metrics(y_true, y_pred, full=False):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    atrasados = y_true > 0

    out = {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae_so_atrasados": mean_absolute_error(y_true[atrasados], y_pred[atrasados]) if atrasados.any() else np.nan,
        "vies_medio": (y_pred - y_true).mean(),
    }
    if full:
        out["r2"] = r2_score(y_true, y_pred)
        out["spearman"] = spearmanr(y_true, y_pred)[0] if np.unique(y_pred).size > 1 else np.nan
    return out


def run_cv(num, cat):
    cols = num + cat
    cv_rows, oof_rows = [], []
    for f in folds:
        tr, va = f["train"], f["val"]
        baseline = make_baseline().fit(tr[cols], tr["target_dias_atraso"])
        ridge = make_ridge(num, cat).fit(tr[cols], tr["target_dias_atraso"])
        preds = {"baseline_mediana": predict(baseline, va, cols), "ridge": predict(ridge, va, cols)}

        for nome, pred in preds.items():
            cv_rows.append({"fold": f["fold"], "modelo": nome, **metrics(va["target_dias_atraso"], pred)})
        oof_rows.append(pd.DataFrame({"fold": f["fold"], "id_contrato": va["idContrato"].values,
                                      "real": va["target_dias_atraso"].values, **preds}))
    return pd.concat([pd.DataFrame(cv_rows)]), pd.concat(oof_rows, ignore_index=True)

In [8]:
cv, oof = run_cv(NUMERIC_FEATURES, CATEGORICAL_FEATURES)

print("MAE por janela:")
display(cv.pivot(index="fold", columns="modelo", values="mae"))

print("Resumo entre janelas:")
cv_resumo = cv.groupby("modelo").agg(mae_medio=("mae", "mean"), mae_dp=("mae", "std"), rmse_medio=("rmse", "mean"))
display(cv_resumo)

print("Métricas agregadas (20 contratos de validação):")
display(pd.DataFrame({n: metrics(oof["real"], oof[n], full=True) for n in ["baseline_mediana", "ridge"]}).T)

MAE por janela:


modelo,baseline_mediana,ridge
fold,,
1,30.000,44.485
2,63.800,64.540
3,0.000,31.487
4,17.800,69.976


Resumo entre janelas:


,mae_medio,mae_dp,rmse_medio
modelo,,,
baseline_mediana,27.900,26.917,52.053
ridge,52.622,17.852,72.621


Métricas agregadas (20 contratos de validação):


,mae,rmse,mae_so_atrasados,vies_medio,r2,spearman
baseline_mediana,27.900,69.787,79.714,-27.900,-0.190,NaN
ridge,52.622,78.438,73.148,19.741,-0.504,0.276


## 7. Teste final

In [9]:
baseline_final = make_baseline().fit(dev[FEATURES], dev["target_dias_atraso"])
ridge_final = make_ridge(NUMERIC_FEATURES, CATEGORICAL_FEATURES).fit(dev[FEATURES], dev["target_dias_atraso"])

test_pred = pd.DataFrame({
    "id_contrato": test["idContrato"].values,
    "real": test["target_dias_atraso"].values,
    "baseline_mediana": predict(baseline_final, test, FEATURES),
    "ridge": predict(ridge_final, test, FEATURES),
})

display(pd.DataFrame({n: metrics(test_pred["real"], test_pred[n], full=True) for n in ["baseline_mediana", "ridge"]}).T)
display(test_pred)

coef = pd.Series(
    ridge_final["model"].coef_, index=ridge_final["prep"].get_feature_names_out()
).sort_values(key=np.abs, ascending=False)
print("Maiores coeficientes (variáveis padronizadas):")
display(coef.head(10).to_frame("coeficiente"))

,mae,rmse,mae_so_atrasados,vies_medio,r2,spearman
baseline_mediana,21.875,54.352,87.500,-21.875,-0.193,NaN
ridge,60.550,111.761,50.029,60.550,-4.045,0.517


,id_contrato,real,baseline_mediana,ridge
0,173,0,0.000,0.000
1,201,0,0.000,303.991
2,287,0,0.000,0.000
3,283,0,0.000,16.582
4,342,0,0.000,41.420
5,317,23,0.000,70.100
6,254,0,0.000,22.351
7,377,152,0.000,204.958


Maiores coeficientes (variáveis padronizadas):


,coeficiente
num__parc_qtd_supressao_historico,-63.129
num__parc_qtd_aditivo_historico,52.928
num__parc_qtd_servicos_originais,-47.603
num__lib_atraso_considerado_soma,42.514
num__adv_qtd,39.304
num__adp_qtd,-35.622
num__lib_abono_atraso_soma,31.240
num__parc_qtd_reajuste,28.235
cat__id_setor_divisao_10,27.658
cat__id_setor_divisao_12,-20.796


## 8. Efeito de cada grupo de tabelas (apenas validação temporal)

Adiciona os grupos um a um, na ordem em que a informação passa a existir. O teste final não é usado aqui.

In [10]:
ordem = ["parcelas", "aditivos_valor", "aditivos_prazo", "liberacoes"]

rows = []
num_atual = list(CONTRATO_NUM)
cenarios = [("contrato", list(num_atual))]
for g in ordem:
    num_atual = num_atual + GRUPOS_AUX[g]
    cenarios.append((f"+ {g}", list(num_atual)))

for nome, num in cenarios:
    cv_g, _ = run_cv(num, CATEGORICAL_FEATURES)
    mae = cv_g.groupby("modelo")["mae"].mean()
    rows.append({
        "dados": nome,
        "n_features_numericas": len(num),
        "mae_baseline": mae["baseline_mediana"],
        "mae_ridge": mae["ridge"],
        "ridge_menos_baseline": mae["ridge"] - mae["baseline_mediana"],
    })

display(pd.DataFrame(rows))

,dados,n_features_numericas,mae_baseline,mae_ridge,ridge_menos_baseline
0,contrato,1,27.900,65.669,37.769
1,+ parcelas,8,27.900,74.424,46.524
2,+ aditivos_valor,9,27.900,71.284,43.384
3,+ aditivos_prazo,11,27.900,76.984,49.084
4,+ liberacoes,15,27.900,52.622,24.722


## 9. Exportação do dataset enriquecido

In [11]:
OUTPUT_DIR = Path("07_dataset_enriquecido_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

colunas_saida = ["idContrato", "NumeroContrato", "data_ordem_servico", "data_recebimento_provisorio",
                 "target_dias_atraso"] + CONTRATO_NUM + CONTRATO_CAT + COLUNAS_AUX

enriquecido[colunas_saida].sort_values("idContrato").to_csv(
    OUTPUT_DIR / "dataset_contratos_enriquecido.csv", index=False, encoding="utf-8-sig"
)
print("Salvo em:", (OUTPUT_DIR / "dataset_contratos_enriquecido.csv").resolve(), "|", len(enriquecido), "linhas,", len(colunas_saida), "colunas")

Salvo em: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\notebooks\07_dataset_enriquecido_outputs\dataset_contratos_enriquecido.csv | 48 linhas, 22 colunas


## Leitura dos resultados

**Mesmo com o dataset enriquecido, o Ridge não supera o baseline.**

| | MAE validação temporal (média de 4 janelas) | MAE teste final (8 contratos) |
|---|---|---|
| Baseline (mediana = 0 dia) | 27,9 | 21,9 |
| Ridge, dataset enriquecido (17 features) | 52,6 | 60,6 |
| Ridge, só colunas do contrato (notebook 06) | 65,7 | 40,8 |

- **Validação:** o Ridge enriquecido melhora em relação ao 06 (65,7 → 52,6), mas continua pior que o baseline nas 4 janelas.
- **Teste:** piora em relação ao 06 (40,8 → 60,6). O principal motivo é o contrato 201, com atraso real 0 e previsão de ~304 dias: com poucos contratos de treino e `alpha = 1`, o Ridge extrapola para contratos grandes.
- **Ordenação:** o Spearman fica positivo (0,28 na validação e 0,52 no teste), um sinal que os notebooks anteriores não tinham. Com 8 contratos e só 2 atrasados, o valor do teste não é conclusivo.
- **Por grupo de tabelas (seção 8):** parcelas, aditivos de valor e aditivos de prazo só pioram o MAE (de 65,7 para 77,0). Liberações é o único grupo que reduz o erro (de 77,0 para 52,6), ainda assim acima do baseline.
- **Origem do pouco sinal:** liberações e aditivos só existem durante ou depois da obra, e `lib_atraso_considerado_soma` e `lib_abono_atraso_soma` já descrevem atraso por parcela. Esse sinal não está disponível na ordem de serviço, então não serve como previsão antecipada.
- **Muitas features para poucos contratos:** são 17 colunas (mais o one-hot) para 20 a 40 contratos de treino. Os coeficientes maiores são de contagens de parcelas e aditivos, e com sinais que mudam de uma janela para outra, sem leitura estável.
- **Cobertura das tabelas:** o contrato 12 não tem parcelas nem liberações; 11 contratos não têm aditivo de prazo e 10 não têm aditivo de valor. Todos entram com 0, o que mistura "não teve" com "não há registro".

**Ideias para testar depois (não executadas):** aumentar `RIDGE_ALPHA` (ou usar `RidgeCV`), reduzir o conjunto a poucos grupos, ou reaplicar o desenho com os dados da obra cortados em um ponto fixo do tempo (como nos snapshots do 04B), o que dá uma previsão utilizável durante a execução.